# NVILA 타일링 · 패치화 · AutoGaze 멀티스케일

이 노트북은 NVILA가 비디오 프레임을 처리하는 전체 과정을 시각화합니다.

---
## 주요 파라미터 (실제 가중치 기준)

| 구성 요소 | 값 |
|---|---|
| ViT 입력 타일 크기 | 392 × 392 px |
| ViT patch_size | 14 × 14 px |
| 타일당 패치 수 | (392/14)² = **28 × 28 = 784** |
| 최대 타일 수 (동영상) | **8** |
| ViT 멀티스케일 | 56 · 112 · 196 · 392 px |
| AutoGaze 입력 크기 | 224 × 224 px |
| AutoGaze scales | 32 · 64 · 112 · 224 px |
| AutoGaze 토큰/프레임 | **265** (스케일별 4+16+49+196) |

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

# 한글 폰트 설정
import matplotlib
import platform
if platform.system() == 'Darwin':
    matplotlib.rc('font', family='AppleGothic')
else:
    for font in ['NanumGothic', 'NanumBarunGothic', 'DejaVu Sans']:
        try:
            matplotlib.rc('font', family=font)
            break
        except:
            continue
matplotlib.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

---
## 1. 공간 타일링 (Spatial Tiling)

입력 프레임을 가로세로 비율에 맞게 최대 8개의 392×392 타일로 분할합니다.

In [ ]:
def draw_tiling(ax, cols, rows, frame_w, frame_h, title, color='steelblue'):
    """프레임을 cols×rows 타일로 분할하는 다이어그램."""
    tile_size = 392
    resized_w = cols * tile_size
    resized_h = rows * tile_size

    ax.set_xlim(-0.5, resized_w + 0.5)
    ax.set_ylim(-0.5, resized_h + 0.5)
    ax.set_aspect('equal')
    ax.axis('off')

    colors = plt.cm.Pastel1(np.linspace(0, 0.8, cols * rows))
    idx = 0
    for r in range(rows):
        for c in range(cols):
            rect = mpatches.FancyBboxPatch(
                (c * tile_size + 4, r * tile_size + 4),
                tile_size - 8, tile_size - 8,
                boxstyle='round,pad=0', linewidth=1.5,
                edgecolor='#333', facecolor=colors[idx]
            )
            ax.add_patch(rect)
            ax.text(
                (c + 0.5) * tile_size, (r + 0.5) * tile_size,
                f'Tile\n{r*cols+c+1}',
                ha='center', va='center', fontsize=9, fontweight='bold'
            )
            idx += 1

    # 눈금
    for c in range(cols + 1):
        ax.axvline(c * tile_size, color='#555', lw=1.5)
    for r in range(rows + 1):
        ax.axhline(r * tile_size, color='#555', lw=1.5)

    ax.set_title(
        f'{title}\n원본 {frame_w}×{frame_h}  →  리사이즈 {resized_w}×{resized_h}  →  {cols}×{rows}={cols*rows}타일',
        fontsize=9, pad=8
    )


examples = [
    (1, 1, 392, 392,   '392×392 (정사각형)'),
    (2, 2, 784, 784,   '784×784 (정사각형)'),
    (2, 1, 784, 392,   '784×392 (2:1 가로)'),
    (4, 2, 3840, 2160, '3840×2160 (4K 16:9)'),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, (cols, rows, fw, fh, title) in zip(axes, examples):
    draw_tiling(ax, cols, rows, fw, fh, title)

fig.suptitle('NVILA 공간 타일링 예시  (타일 크기 392×392 고정)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("★ max_tiles_video=8: 4K도 최대 8타일 (4×2)")
print("★ 비율 보존: 원본 비율에 가장 가까운 (cols, rows) 조합 선택")
print("★ 별도 thumbnail: 프레임 전체를 392×392 단일 이미지로 다운샘플 → 큰 그림 맥락 제공")

---
### 1-b. 구체적 시각화 — 합성 이미지로 보는 타일링 & 패치화

추상 다이어그램 대신, 실제 이미지처럼 보이는 합성 장면을 사용해
**타일 경계**, **패치 그리드**, **개별 패치 좌표**를 직접 확인합니다.  
텍스트(Unicode 박스) 버전도 함께 출력합니다.

In [ ]:

# ════════════════════════════════════════════════════════════
# 구체적 시각화 — 합성 이미지로 보는 타일링 & 패치화
# ════════════════════════════════════════════════════════════
TILE = 392
PATCH = 14
N_PATCH = TILE // PATCH  # 28

def _make_scene(w, h):
    """하늘·건물·사람 합성 장면 (numpy only, 0-1 float)."""
    img = np.zeros((h, w, 3), dtype=np.float32)
    h2 = h // 2
    img[:h2] = [0.32, 0.52, 0.94]          # 하늘
    img[h2:] = [0.22, 0.52, 0.22]          # 땅
    bx1 = w // 8;  bx2 = bx1 + max(2, w // 7)
    img[h // 4:, bx1:bx2] = [0.52, 0.52, 0.60]  # 건물
    pw, ph = max(2, w // 32), max(2, h // 14)
    for wy in range(h // 4 + max(1, h // 18), h - ph, max(1, h // 10)):
        for wx in range(bx1 + max(1, w // 40), bx2 - pw, max(1, w // 20)):
            img[wy:wy+ph, wx:wx+pw] = [1.0, 0.88, 0.25]  # 창문
    px = int(w * 0.6);  py = int(h * 0.35)
    tw, th = max(2, w // 22), max(2, h // 5)
    img[py:py+th, px:px+tw] = [0.78, 0.35, 0.22]         # 몸
    hh = max(2, h // 12)
    img[py-hh:py, px:px+tw] = [0.93, 0.76, 0.54]         # 머리
    return np.clip(img, 0, 1)

def _nn_resize(img, nw, nh):
    """최근접 이웃 리사이즈 (PIL 없이 numpy)."""
    h, w = img.shape[:2]
    xi = (np.arange(nw) * w / nw).astype(int).clip(0, w - 1)
    yi = (np.arange(nh) * h / nh).astype(int).clip(0, h - 1)
    return img[np.ix_(yi, xi)]

# ── matplotlib 시각화 (두 해상도) ────────────────────────────
for W, H, COLS, ROWS, lbl in [
    (784,  392,  2, 1, '784×392  →  2×1 타일'),
    (3840, 2160, 4, 2, '4K  3840×2160  →  4×2 타일'),
]:
    RW, RH   = COLS * TILE, ROWS * TILE
    frame    = _make_scene(W, H)
    resized  = _nn_resize(frame, RW, RH)
    tile1    = resized[:TILE, :TILE].copy()

    fig, axes = plt.subplots(1, 4, figsize=(17, 5))
    for ax in axes:
        ax.axis('off')

    # ① 원본 프레임
    dw = min(W, 600);  dh = max(1, round(dw * H / W))
    axes[0].imshow(_nn_resize(frame, dw, dh), origin='upper')
    axes[0].set_title(f'① 원본 프레임\n{W} × {H} px', fontsize=10, fontweight='bold')

    # ② 리사이즈 + 타일 경계
    ax = axes[1]
    ax.imshow(resized, origin='upper')
    for c in range(COLS + 1):
        ax.axvline(c * TILE - 0.5, color='#ff2222', lw=2.5)
    for r in range(ROWS + 1):
        ax.axhline(r * TILE - 0.5, color='#ff2222', lw=2.5)
    for r in range(ROWS):
        for c in range(COLS):
            ax.text((c + 0.5)*TILE, (r + 0.5)*TILE, f'T{r*COLS+c+1}',
                    ha='center', va='center', fontsize=13, fontweight='bold',
                    color='white',
                    bbox=dict(facecolor='#cc0000', alpha=0.75, boxstyle='round,pad=0.3'))
    ax.set_title(f'② 타일 분할\n{RW}×{RH} → {COLS}×{ROWS}={COLS*ROWS}타일  (각 392×392)',
                 fontsize=10, fontweight='bold')

    # ③ Tile 1 + 패치 그리드 오버레이
    ax = axes[2]
    ax.imshow(tile1, origin='upper')
    for p in range(0, TILE + 1, PATCH):
        ax.axhline(p - 0.5, color='yellow', lw=0.6, alpha=0.85)
        ax.axvline(p - 0.5, color='yellow', lw=0.6, alpha=0.85)
    ax.set_title(f'③ Tile 1  (392×392)\n패치 그리드 14px:  {N_PATCH}×{N_PATCH} = 784 patches',
                 fontsize=10, fontweight='bold')
    ax.text(3, 12, f'픽셀 범위: x[0:{TILE}], y[0:{TILE}]',
            fontsize=7, color='white',
            bbox=dict(facecolor='black', alpha=0.55, pad=2))

    # ④ 패치 6×6 확대
    ax = axes[3]
    Z = 6;  zpx = Z * PATCH
    zoom = _nn_resize(tile1[:zpx, :zpx], 288, 288)
    ax.imshow(zoom, origin='upper')
    ep = 288 / Z
    for p in range(Z + 1):
        ax.axhline(p * ep - 0.5, color='yellow', lw=1.5, alpha=0.9)
        ax.axvline(p * ep - 0.5, color='yellow', lw=1.5, alpha=0.9)
    for pr in range(Z):
        for pc in range(Z):
            ax.text((pc + 0.5)*ep, (pr + 0.5)*ep, f'P({pr},{pc})',
                    ha='center', va='center', fontsize=6.5, color='white', alpha=0.95)
    ax.add_patch(mpatches.Rectangle((ep, ep), 4*ep, 4*ep,
                 facecolor='cyan', alpha=0.20, edgecolor='cyan', lw=2))
    ax.set_title(f'④ 패치 확대  (좌상단 {Z}×{Z})\n각 패치 = 14×14 px  →  1 ViT 토큰',
                 fontsize=10, fontweight='bold')

    fig.suptitle(f'타일링 & 패치화 구체적 시각화:  {lbl}',
                 fontsize=12, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.show()

# ── 텍스트(ASCII) 기반 타일 구조 ────────────────────────────
def _ascii_tile_grid(cols, rows, cell_w=10):
    """Unicode 박스 그리기 문자로 타일 그리드 출력."""
    sep = '─' * cell_w
    lines = []
    for r in range(rows):
        lines.append(('┌' if r == 0 else '├') +
                     ('┬' if r == 0 else '┼').join([sep] * cols) +
                     ('┐' if r == 0 else '┤'))
        row = '│' + '│'.join([f'Tile {r*cols+c+1:02d}'.center(cell_w) for c in range(cols)]) + '│'
        lines.append(row)
        sub = '│' + '│'.join([f'392×392'.center(cell_w)] * cols) + '│'
        lines.append(sub)
    lines.append('└' + '┴'.join([sep] * cols) + '┘')
    return '\n'.join(lines)

print('=' * 62)
print('  타일 구조  (텍스트)')
print('=' * 62)
for name, cols, rows, w, h in [
    ('392×392  →  1×1', 1, 1, 392, 392),
    ('784×784  →  2×2', 2, 2, 784, 784),
    ('784×392  →  2×1', 2, 1, 784, 392),
    ('4K 3840×2160 → 4×2', 4, 2, 3840, 2160),
]:
    print(f'\n  입력: {name}')
    for line in _ascii_tile_grid(cols, rows).split('\n'):
        print(f'  {line}')

print()
print('=' * 62)
print('  패치 구조  (한 타일 내부, 좌상단 5×5만 표시)')
print('=' * 62)
pw = 9;  sep_p = '─' * pw
for r in range(5):
    print('  ' + ('┌' if r == 0 else '├') +
          ('┬' if r == 0 else '┼').join([sep_p] * 5) +
          ('┐' if r == 0 else '┤') + '  ···')
    print('  │' + '│'.join([f'P({r:02d},{c:02d})'.center(pw) for c in range(5)]) + '│  ···')
print('  └' + '┴'.join([sep_p] * 5) + '┘')
print('   ·' + ' ' * (pw * 5 + 4) + '⋱')
print(f'\n  총 28×28 = 784 패치 / 타일  (각 14×14 px = 1 ViT 토큰)')
print()
print('=' * 62)
print('  요약: 프레임 → 타일 → 패치 → LLM 토큰')
print('=' * 62)
print('''
  입력 프레임 (임의 H×W)
      │
      ▼  비율에 맞게 392 배수로 리사이즈
  cols × rows 타일  (각 392×392 px,  최대 8타일)
      │
      ▼  14px 패치로 분할
  28×28 = 784 패치 / 타일
      │
      ├─▶ SigLIP ViT  →  784 토큰 / 타일
      │
      └─▶ AutoGaze    →  265 gaze 점수 / 타일
              │
              ▼  ratio 적용 (상위 r% 선택)
          784 × ratio 토큰  →  LLM 입력
''')


---
## 2. 타일 → 패치화 (Patch Embedding)

각 392×392 타일은 ViT(SigLIP)에 들어가기 전에 **14×14 px 패치**로 잘립니다.  
타일 1개당 **28×28 = 784 패치**, 8타일이면 **최대 6,272 패치**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# ── 왼쪽: 타일 하나를 28×28 패치로 분할 ──────────────────────────
ax = axes[0]
n = 28
cmap = plt.cm.YlOrRd
# 중앙에 관심 영역이 있다고 가정 (가우시안 가중치)
cx, cy = 14, 10
yy, xx = np.mgrid[:n, :n]
heat = np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * 5**2))
ax.imshow(heat, cmap='YlOrRd', origin='lower', extent=[0, n, 0, n], alpha=0.6)

for i in range(n + 1):
    ax.axhline(i, color='#888', lw=0.3)
    ax.axvline(i, color='#888', lw=0.3)
ax.set_xlim(0, n); ax.set_ylim(0, n)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title('타일 1장 (392×392)\n→ 28×28 = 784 패치', fontsize=11, fontweight='bold')
ax.text(n/2, -1.8, 'patch_size = 14×14 px  |  392/14 = 28', ha='center', fontsize=9, color='#444')

# ── 가운데: 스케일별 ViT 특징 맵 크기 ────────────────────────────
ax = axes[1]
scales = [56, 112, 196, 392]
patch_size = 14
tokens = [(s // patch_size)**2 for s in scales]
grid_n = [s // patch_size for s in scales]
colors_bar = ['#d4e6f1', '#85c1e9', '#2e86c1', '#1a5276']
bars = ax.barh(range(4), tokens, color=colors_bar, edgecolor='#333', height=0.6)
for i, (t, g) in enumerate(zip(tokens, grid_n)):
    ax.text(t + 5, i, f'{g}×{g} = {t} 토큰', va='center', fontsize=10)
ax.set_yticks(range(4))
ax.set_yticklabels([f'Scale {s}px' for s in scales], fontsize=10)
ax.set_xlabel('ViT 토큰 수 (타일 1개 기준)', fontsize=10)
ax.set_title('ViT 멀티스케일 특징 맵\n(타일 1개당)', fontsize=11, fontweight='bold')
ax.set_xlim(0, 900)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# ── 오른쪽: 타일 수별 총 토큰 수 ─────────────────────────────────
ax = axes[2]
n_tiles = [1, 2, 4, 6, 8]
# NVILA는 392 scale 하나만 LLM에 전달 (784 토큰/타일)
total_tokens = [t * 784 for t in n_tiles]
bars2 = ax.bar(n_tiles, total_tokens, color='#5dade2', edgecolor='#1a5276', width=0.7)
for x, y in zip(n_tiles, total_tokens):
    ax.text(x, y + 30, str(y), ha='center', fontsize=10, fontweight='bold')
ax.axhline(total_tokens[-1], color='#e74c3c', lw=1.5, ls='--', alpha=0.7)
ax.text(8.4, total_tokens[-1], '4K 상한', color='#e74c3c', va='center', fontsize=9)
ax.set_xlabel('공간 타일 수', fontsize=10)
ax.set_ylabel('LLM 입력 시각 토큰 수', fontsize=10)
ax.set_title('타일 수 → 시각 토큰 수\n(392px scale 기준)', fontsize=11, fontweight='bold')
ax.set_xticks(n_tiles)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f"타일 1개 → 784 패치 → ViT 처리 → LLM에 784 토큰 전달")
print(f"4K(8타일) = {8*784:,} 시각 토큰  vs  thumbnail 추가 = +784 토큰")

---
## 3. AutoGaze 멀티스케일 처리

AutoGaze는 ViT와 **별도의** 경량 모델입니다.  
각 타일(392×392)을 **224×224로 리사이즈** 후 처리하며,  
**4개 스케일(32·64·112·224)**에서 gaze 점수를 예측합니다.

```
스케일    그리드    토큰 수   역할
  32      2×2        4      전체 구도 파악 (매우 거친 attention)
  64      4×4       16      중간 영역 분할
 112      7×7       49      세부 영역 분할
 224     14×14     196      패치 수준 세밀한 gaze (ViT와 1:1 대응)
────────────────────────────────
합계                265      총 gaze 토큰/프레임
```

이 4스케일 gaze map을 ViT의 target_scales `[56, 112, 196, 392]`에 **bilinear interpolation**으로 맵핑한 뒤,  
각 ViT 패치 위치의 중요도(gaze score)를 계산합니다.

In [ ]:
fig = plt.figure(figsize=(16, 7))
gs = gridspec.GridSpec(2, 5, figure=fig, hspace=0.5, wspace=0.4)

ag_scales = [32, 64, 112, 224]
ag_grids  = [2, 4, 7, 14]
ag_tokens = [4, 16, 49, 196]
ag_colors = ['#fadbd8', '#f1948a', '#e74c3c', '#922b21']
scale_labels = ['2×2 (구도 파악)', '4×4 (영역 분할)', '7×7 (세부 영역)', '14×14 (패치 수준)']

np.random.seed(42)

for col, (sc, gn, tok, clr, lbl) in enumerate(zip(ag_scales, ag_grids, ag_tokens, ag_colors, scale_labels)):
    ax = fig.add_subplot(gs[0, col])

    # 가우시안 gaze 히트맵
    cx, cy = gn * 0.55, gn * 0.4
    sig = gn * 0.25
    yy, xx = np.mgrid[:gn, :gn]
    heat = np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * sig**2))
    heat += 0.08 * np.random.rand(gn, gn)
    heat /= heat.max()

    ax.imshow(heat, cmap='Reds', vmin=0, vmax=1, origin='lower',
              extent=[0, gn, 0, gn], interpolation='nearest')
    for i in range(gn + 1):
        ax.axhline(i, color='white', lw=0.5)
        ax.axvline(i, color='white', lw=0.5)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'Scale {sc}px\n{lbl}\n({tok} 토큰)', fontsize=8.5, fontweight='bold')

# 오른쪽: 총 토큰 수 요약 막대
ax_sum = fig.add_subplot(gs[0, 4])
ax_sum.barh(range(4), ag_tokens, color=ag_colors, edgecolor='#555', height=0.6)
for i, t in enumerate(ag_tokens):
    ax_sum.text(t + 1, i, str(t), va='center', fontsize=10, fontweight='bold')
ax_sum.set_yticks(range(4))
ax_sum.set_yticklabels([f'{s}px' for s in ag_scales], fontsize=9)
ax_sum.set_xlabel('토큰 수', fontsize=9)
ax_sum.set_title(f'스케일별 토큰\n합계={sum(ag_tokens)}', fontsize=9, fontweight='bold')
ax_sum.spines['top'].set_visible(False); ax_sum.spines['right'].set_visible(False)

# ── 하단: ViT target_scales 맵핑 다이어그램 ──────────────────────
ax_map = fig.add_subplot(gs[1, :])
ax_map.axis('off')

vit_scales  = [56, 112, 196, 392]
vit_grids   = [4, 8, 14, 28]    # target_scale // patch_size(14)
vit_tokens  = [16, 64, 196, 784]

xs_ag  = [0.08, 0.27, 0.46, 0.65]  # AutoGaze boxes x
xs_vit = [0.08, 0.27, 0.46, 0.65]  # ViT boxes x
y_ag, y_vit = 0.78, 0.22
bw, bh = 0.14, 0.22

ag_box_colors  = ag_colors
vit_box_colors = ['#d6eaf8', '#7fb3d3', '#2e86c1', '#1a5276']

for i, (x, sc, tok, clr) in enumerate(zip(xs_ag, ag_scales, ag_tokens, ag_box_colors)):
    rect = mpatches.FancyBboxPatch((x, y_ag - bh/2), bw, bh,
        boxstyle='round,pad=0.01', facecolor=clr, edgecolor='#555', lw=1.2,
        transform=ax_map.transAxes, clip_on=False)
    ax_map.add_patch(rect)
    ax_map.text(x + bw/2, y_ag, f'AG {sc}px\n{ag_grids[i]}×{ag_grids[i]}\n{tok}tok',
                ha='center', va='center', fontsize=8, fontweight='bold',
                transform=ax_map.transAxes)

for i, (x, sc, tok, clr) in enumerate(zip(xs_vit, vit_scales, vit_tokens, vit_box_colors)):
    rect = mpatches.FancyBboxPatch((x, y_vit - bh/2), bw, bh,
        boxstyle='round,pad=0.01', facecolor=clr, edgecolor='#555', lw=1.2,
        transform=ax_map.transAxes, clip_on=False)
    ax_map.add_patch(rect)
    ax_map.text(x + bw/2, y_vit, f'ViT {sc}px\n{vit_grids[i]}×{vit_grids[i]}\n{tok}tok',
                ha='center', va='center', fontsize=8, fontweight='bold', color='white',
                transform=ax_map.transAxes)

# 화살표: AG → ViT (bilinear interp)
for x in xs_ag:
    ax_map.annotate('', xy=(x + bw/2, y_vit + bh/2 + 0.02),
                    xytext=(x + bw/2, y_ag - bh/2 - 0.02),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='#e67e22', lw=2))

ax_map.text(0.5, 0.5, 'bilinear interpolation\n(gaze map 크기 적응)',
            ha='center', va='center', fontsize=9, color='#e67e22', style='italic',
            transform=ax_map.transAxes)
ax_map.text(0.02, y_ag, 'AutoGaze\n출력', ha='center', va='center', fontsize=9,
            fontweight='bold', transform=ax_map.transAxes)
ax_map.text(0.02, y_vit, 'ViT 패치\n대응', ha='center', va='center', fontsize=9,
            fontweight='bold', transform=ax_map.transAxes)

fig.suptitle('AutoGaze 멀티스케일 Gaze 예측 & ViT 패치 맵핑', fontsize=13, fontweight='bold', y=1.01)
plt.show()

---
## 4. Gaze 기반 패치 선택 (ratio 효과)

`gazing_ratio`는 AutoGaze가 예측한 점수 상위 **r%** 패치만 LLM에 전달하는 threshold입니다.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
ratios = [0.2, 0.5, 0.75, 1.0]

np.random.seed(7)
n = 14  # AutoGaze 최대 그리드 (224px scale)

# 관심 영역이 있는 gaze 맵 생성
cx, cy = 8, 6
yy, xx = np.mgrid[:n, :n]
heat = np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * 3**2))
heat += 0.15 * np.random.rand(n, n)
heat /= heat.max()

total_patches = n * n
scores_flat = heat.flatten()

for ax, ratio in zip(axes, ratios):
    k = max(1, int(total_patches * ratio))
    threshold = np.sort(scores_flat)[::-1][k - 1]

    selected = (heat >= threshold).astype(float)
    display = np.zeros((*heat.shape, 4))

    # 선택된 패치: 점수에 따른 붉은 계열
    for r in range(n):
        for c in range(n):
            if selected[r, c]:
                intensity = heat[r, c]
                display[r, c] = [1, 1 - intensity * 0.8, 1 - intensity * 0.8, 1.0]
            else:
                display[r, c] = [0.85, 0.85, 0.85, 1.0]  # 회색: 미선택

    ax.imshow(display, origin='lower', extent=[0, n, 0, n], interpolation='nearest')
    for i in range(n + 1):
        ax.axhline(i, color='white', lw=0.4)
        ax.axvline(i, color='white', lw=0.4)

    ax.set_xticks([]); ax.set_yticks([])
    label = 'AutoGaze OFF (전체)' if ratio == 1.0 else f'ratio = {ratio}'
    ax.set_title(f'{label}\n선택 패치: {k}/{total_patches} ({ratio*100:.0f}%)',
                 fontsize=10, fontweight='bold')

    # 범례
    from matplotlib.patches import Patch
    legend = [
        Patch(facecolor='#e74c3c', label='선택 (gazed)'),
        Patch(facecolor='#d5d8dc', label='미선택'),
    ]
    ax.legend(handles=legend, loc='lower right', fontsize=7, framealpha=0.8)

fig.suptitle('Gazing Ratio별 패치 선택 (14×14 그리드, 한 타일 기준)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("ratio=1.0 → AutoGaze 모델 자체를 호출하지 않음 (모든 패치 사용)")
print("ratio<1.0 → AutoGaze 항상 동일하게 실행, 점수 상위 r%만 LLM에 전달")

---
## 5. 전체 파이프라인

비디오 프레임부터 LLM 입력까지 전체 흐름을 한눈에 정리합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.set_xlim(0, 16); ax.set_ylim(0, 6)
ax.axis('off')

def box(ax, x, y, w, h, label, sublabel='', color='#d6eaf8', fontsize=10):
    rect = mpatches.FancyBboxPatch((x, y), w, h,
        boxstyle='round,pad=0.15', facecolor=color, edgecolor='#2c3e50', lw=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2 + (0.12 if sublabel else 0), label,
            ha='center', va='center', fontsize=fontsize, fontweight='bold')
    if sublabel:
        ax.text(x + w/2, y + h/2 - 0.28, sublabel,
                ha='center', va='center', fontsize=8, color='#555')

def arrow(ax, x1, y, x2, label=''):
    ax.annotate('', xy=(x2, y), xytext=(x1, y),
                arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2))
    if label:
        ax.text((x1+x2)/2, y + 0.18, label, ha='center', fontsize=8, color='#555')

def split_arrow(ax, x1, y_mid, x2, y_top, y_bot, label=''):
    """분기 화살표"""
    ax.plot([x1, x1], [y_top, y_bot], color='#2c3e50', lw=1.5)
    ax.annotate('', xy=(x2, y_top), xytext=(x1, y_top),
                arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.5))
    ax.annotate('', xy=(x2, y_bot), xytext=(x1, y_bot),
                arrowprops=dict(arrowstyle='->', color='#e67e22', lw=1.5))

# Stage 1: 입력 프레임
box(ax, 0.1, 2.3, 1.8, 1.4, '입력 프레임', 'H×W (임의 해상도)', '#fdebd0')

# Stage 2: 타일링
arrow(ax, 1.9, 3.0, 2.5)
box(ax, 2.5, 2.3, 1.8, 1.4, '공간 타일링', f'≤8 tiles\n392×392 each', '#d5e8d4')
ax.text(2.5 + 0.9, 2.0, '+ thumbnail\n(392×392)', ha='center', fontsize=7.5, color='#888')

# Stage 3: 분기 (ViT path + AutoGaze path)
arrow(ax, 4.3, 3.0, 4.9)
ax.plot([4.9, 4.9], [3.6, 2.4], color='#2c3e50', lw=1.5)

# ViT path (위)
ax.annotate('', xy=(5.5, 3.7), xytext=(4.9, 3.7),
            arrowprops=dict(arrowstyle='->', color='#2e86c1', lw=2))
box(ax, 5.5, 3.1, 2.2, 1.2, 'SigLIP ViT', '784 패치/타일\n(14×14 px)', '#d6eaf8')

# AutoGaze path (아래)
ax.annotate('', xy=(5.5, 2.3), xytext=(4.9, 2.3),
            arrowprops=dict(arrowstyle='->', color='#e67e22', lw=2))
box(ax, 5.5, 1.7, 2.2, 1.2, 'AutoGaze', '4-scale gaze\n265 tok/frame', '#fdebd0')

# Stage 4: AutoGaze → gaze score
arrow(ax, 7.7, 2.3, 8.3, '점수 맵핑')
box(ax, 8.3, 1.7, 2.0, 1.2, 'Patch 선택', f'ratio × 784\n= top-k 패치', '#f9ebea')

# ViT → token
arrow(ax, 7.7, 3.7, 8.3)
box(ax, 8.3, 3.1, 2.0, 1.2, 'ViT 토큰', '784 tok/tile\n→ LLM dim', '#d6eaf8')

# Gaze mask → ViT token masking
ax.annotate('', xy=(9.3, 3.1), xytext=(9.3, 2.9),
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2))
ax.text(9.5, 3.0, 'gaze\nmask', ha='left', va='center', fontsize=8, color='#e74c3c')

# Stage 5: 선택된 ViT 토큰
arrow(ax, 10.3, 3.7, 11.0)
box(ax, 11.0, 3.1, 2.2, 1.2, '선택된 시각 토큰', f'784×ratio\n× n_tiles', '#d5e8d4')

# Stage 6: LLM
arrow(ax, 13.2, 3.7, 13.8)
box(ax, 13.8, 2.5, 2.0, 2.0, 'LLM\n(Qwen2)', 'prefill:\n시각 + 텍스트 토큰', '#e8daef', fontsize=11)

# 범례
ax.text(0.1, 5.5, '파란색: ViT 경로', color='#2e86c1', fontsize=9, fontweight='bold')
ax.text(3.5, 5.5, '주황색: AutoGaze 경로', color='#e67e22', fontsize=9, fontweight='bold')
ax.text(7.0, 5.5, '빨간색: gaze mask 적용', color='#e74c3c', fontsize=9, fontweight='bold')

fig.suptitle('NVILA 전체 파이프라인 (비디오 프레임 → LLM 입력)',
             fontsize=13, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

---
## 6. 해상도별 토큰 수 정리

실제 입력 해상도가 다를 때 LLM에 전달되는 시각 토큰 수를 계산합니다.

In [ ]:
import math

def find_tiles(w, h, max_tiles=8, tile_size=392):
    """가로세로 비율에 맞는 최적 (cols, rows) 반환."""
    aspect = w / h
    best, best_diff = (1, 1), float('inf')
    for n in range(1, max_tiles + 1):
        for c in range(1, n + 1):
            r = n // c
            if c * r > max_tiles: continue
            diff = abs(aspect - c / r)
            if diff < best_diff or (diff == best_diff and c*r > best[0]*best[1]):
                best_diff, best = diff, (c, r)
    cols, rows = best
    return cols, rows, cols * rows

cases = [
    ('392×392',   392,  392),
    ('784×784',   784,  784),
    ('784×392',   784,  392),
    ('1920×1080', 1920, 1080),
    ('4K 3840×2160', 3840, 2160),
    ('8K 7680×4320', 7680, 4320),
]

ratios_to_check = [0.25, 0.5, 0.75, 1.0]

print(f"{'해상도':<18} {'tiles':>6} {'토큰/타일':>9} ", end='')
for r in ratios_to_check:
    print(f"{'ratio='+str(r):>12}", end='')
print()
print('-' * 80)

for name, w, h in cases:
    cols, rows, n_tiles = find_tiles(w, h)
    tokens_per_tile = 784  # 28×28
    total_base = n_tiles * tokens_per_tile
    print(f"{name:<18} {n_tiles:>4}({cols}×{rows}) {tokens_per_tile:>9}", end='')
    for r in ratios_to_check:
        sel = max(1, int(total_base * r))
        print(f"{sel:>12,}", end='')
    print()

print()
print("※ thumbnail (+784 토큰) 은 항상 추가, 위 표에 미포함")
print("※ 8K도 max_tiles=8 상한에 걸려 4K와 동일한 타일 수")